In [ ]:
from pyspark.sql.functions import *

In [ ]:
catalog = dbutils.widgets.get("catalog")

In [ ]:
%py
df_products_silver = spark.sql(f"""
    WITH CTE_total_stock AS(
        SELECT 
        product_id
        ,SUM(quantity) AS quantity_total_stock
        FROM {catalog}.bronze.stocks
        GROUP BY
        product_id
    )
    SELECT
     a.product_id
    ,a.product_name
    ,a.brand_id
    ,c.brand_name
    ,a.category_id
    ,b.category_name
    ,a.model_year
    ,a.list_price
    ,e.quantity_total_stock AS total_stock
    FROM {catalog}.bronze.products a
    LEFT JOIN {catalog}.bronze.categories b
    ON a.category_id = b.category_id
    LEFT JOIN {catalog}.bronze.brands c
    ON a.brand_id = c.brand_id
    LEFT JOIN CTE_total_stock e
    ON a.product_id = e.product_id;
""")

In [ ]:
%py
df_products_silver.write.mode("overwrite").format("delta").options(
    overwriteSchema = "true"
).saveAsTable(f"{catalog}.silver.products")